<a href="https://colab.research.google.com/github/zhuzihan728/Image-Restore/blob/main/dehazeFormer_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup
- First, in the **Runtime** menu -> **Change runtime type**, make sure to have ```Hardware Accelerator = GPU```
- Clone repo and install dependencies.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# git clone this repository
!git clone https://github.com/IDKiro/DehazeFormer.git
%cd DehazeFormer

# conda create -n pt1102 python=3.7
# conda activate pt1102

# conda install pytorch=1.10.2 torchvision torchaudio cudatoolkit=11.3 -c pytorch
!pip install opencv-python tqdm pytorch-msssim timm

Cloning into 'DehazeFormer'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 94 (delta 17), reused 11 (delta 11), pack-reused 70 (from 2)
Receiving objects: 100% (94/94), 768.88 KiB | 2.96 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/DehazeFormer


# 2. Download Pre-trained Models


In [ ]:

!cp -r /content/drive/MyDrive/Chris/Dehazeformer/saved_models /content/DehazeFormer/

# 3. Inference

In [ ]:
import torch
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage import img_as_ubyte
import cv2
import json
import os
from tqdm import tqdm
import numpy as np
from PIL import Image
from collections import OrderedDict

class EvalDataset:
    def __init__(self, corrupted_dir, original_dir, mask_dir, metadata_path,
                 im_size=None, transform=None):
        """
        :param corrupted_dir: path to corrupted images folder
        :param original_dir: path to original images folder
        :param mask_dir: path to mask images folder
        :param metadata_path: path to metadata.json
        :param im_size: target size (h, w) or None to keep original
        """
        self.corrupted_dir = corrupted_dir
        self.original_dir = original_dir
        self.mask_dir = mask_dir
        self.im_size = im_size
        self.transform = transform

        with open(metadata_path, 'r') as f:
            self.metadata = json.load(f)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        meta = self.metadata[idx]

        # Load images using names from metadata
        corrupted = Image.open(os.path.join(self.corrupted_dir, meta['corrupted_image']))
        original = Image.open(os.path.join(self.original_dir, meta['original_image']))

        # Resize if needed
        if self.im_size:
            corrupted = corrupted.resize(self.im_size, Image.Resampling.LANCZOS)
            original = original.resize(self.im_size, Image.Resampling.LANCZOS)

        if self.transform:
            corrupted = self.transform(corrupted)
            original = self.transform(original)

        return corrupted, original, meta


class DehazeFormerEvaluator:
    def __init__(self, model, eval_dataset, output_dir, task):
        self.model = model
        self.dataset = eval_dataset
        self.output_dir = os.path.join(output_dir, 'dehazeFormer', task)
        os.makedirs(os.path.join(self.output_dir, 'restored'), exist_ok=True)

    def rgb_to_y(self, img):
        """Convert RGB to Y channel - matches MATLAB rgb2ycbcr"""
        from skimage.color import rgb2ycbcr
        img_ycbcr = rgb2ycbcr(img)  # Expects float [0,1] or uint8 [0,255]
        return img_ycbcr[:, :, 0]

    def evaluate(self):
        results = []

        with torch.no_grad():
            for idx in tqdm(range(len(self.dataset))):
                torch.cuda.ipc_collect()
                torch.cuda.empty_cache()

                # Load images
                corrupted, original, meta = self.dataset[idx]

                # Convert PIL to numpy if needed
                if hasattr(corrupted, 'convert'):
                    corrupted_np = np.array(corrupted.convert('RGB'))
                    original_np = np.array(original.convert('RGB'))
                else:
                    corrupted_np = corrupted
                    original_np = original

                # Prepare input for DehazeFormer (expects range [-1, 1])
                input_ = torch.from_numpy(corrupted_np).float().div(255.).permute(2,0,1).unsqueeze(0).cuda()
                input_ = input_ * 2 - 1  # [0, 1] to [-1, 1]

                # Restore
                restored = self.model(input_).clamp(-1, 1)

                # [-1, 1] to [0, 1]
                restored = restored * 0.5 + 0.5

                # Convert to numpy
                restored_np = restored.permute(0, 2, 3, 1).cpu().detach().numpy()
                restored_np = img_as_ubyte(restored_np[0])

                # Calculate RGB metrics
                psnr_rgb = peak_signal_noise_ratio(original_np, restored_np, data_range=255)
                ssim_rgb = structural_similarity(original_np, restored_np, channel_axis=2, data_range=255)
                mae = np.mean(np.abs(original_np.astype(float) - restored_np.astype(float)))
                mse = np.mean((original_np.astype(float) - restored_np.astype(float)) ** 2)

                # Calculate Y channel metrics
                original_y = self.rgb_to_y(original_np)
                restored_y = self.rgb_to_y(restored_np)
                psnr_y = peak_signal_noise_ratio(original_y, restored_y, data_range=255)
                ssim_y = structural_similarity(original_y, restored_y, data_range=255)

                # Save restored image
                filename = meta['corrupted_image'].replace('_alpha', '_restored_alpha')
                cv2.imwrite(
                    os.path.join(self.output_dir, 'restored', filename),
                    cv2.cvtColor(restored_np, cv2.COLOR_RGB2BGR)
                )

                # Store results
                result = {
                    **meta,
                    'psnr_rgb': float(psnr_rgb),
                    'ssim_rgb': float(ssim_rgb),
                    'psnr_y': float(psnr_y),
                    'ssim_y': float(ssim_y),
                    'mae': float(mae),
                    'mse': float(mse),
                    'restored_image': filename
                }
                results.append(result)

        # Save metrics
        with open(os.path.join(self.output_dir, 'eval_results.json'), 'w') as f:
            json.dump(results, f, indent=2)

        # Print summary
        print(f"\n=== Evaluation Results ===")
        print(f"Average PSNR (RGB): {np.mean([r['psnr_rgb'] for r in results]):.2f} dB")
        print(f"Average SSIM (RGB): {np.mean([r['ssim_rgb'] for r in results]):.4f}")
        print(f"Average PSNR (Y):   {np.mean([r['psnr_y'] for r in results]):.2f} dB")
        print(f"Average SSIM (Y):   {np.mean([r['ssim_y'] for r in results]):.4f}")
        print(f"Average MAE:        {np.mean([r['mae'] for r in results]):.2f}")
        print(f"Average MSE:        {np.mean([r['mse'] for r in results]):.2f}")

        avg_metrics = []
        alpha_ranges = list(set(str(r['alpha_range']) for r in results))
        alpha_ranges.sort()
        latex_string = ""
        for alpha_range in alpha_ranges:
            alpha_results = [r for r in results if str(r['alpha_range']) == alpha_range]
            avg_metrics_alpha = {
                'alpha_range': alpha_range,
                'count': len(alpha_results),
                'avg_psnr (RGB)': float(np.mean([r['psnr_rgb'] for r in alpha_results])),
                'avg_ssim (RGB)': float(np.mean([r['ssim_rgb'] for r in alpha_results])),
                'avg_psnr (Y)': float(np.mean([r['psnr_y'] for r in alpha_results])),
                'avg_ssim (Y)': float(np.mean([r['ssim_y'] for r in alpha_results])),
                'avg_mae': float(np.mean([r['mae'] for r in alpha_results])),
                'avg_mse': float(np.mean([r['mse'] for r in alpha_results]))
            }
            avg_metrics.append(avg_metrics_alpha)
            print(f"======= Alpha {alpha_range} =======")
            print(f"PSNR (RGB): {avg_metrics_alpha['avg_psnr (RGB)']:.2f}")
            print(f"SSIM (RGB): {avg_metrics_alpha['avg_ssim (RGB)']:.4f}")
            print(f"PSNR (Y):   {avg_metrics_alpha['avg_psnr (Y)']:.2f}")
            print(f"SSIM (Y):   {avg_metrics_alpha['avg_ssim (Y)']:.4f}")
            print(f"MAE:        {avg_metrics_alpha['avg_mae']:.2f}")
            print(f"MSE:        {avg_metrics_alpha['avg_mse']:.2f}")
            latex_string += f"{avg_metrics_alpha['avg_psnr (RGB)']:.2f} & {avg_metrics_alpha['avg_ssim (RGB)']:.4f} & {avg_metrics_alpha['avg_psnr (Y)']:.2f} & {avg_metrics_alpha['avg_ssim (Y)']:.4f} & {avg_metrics_alpha['avg_mae']:.2f} & {avg_metrics_alpha['avg_mse']:.2f} &"
        avg_metrics_total = {
            'alpha_range': 'total',
            'count': len(results),
            'avg_psnr': float(np.mean([r['psnr_rgb'] for r in results])),
            'avg_ssim': float(np.mean([r['ssim_rgb'] for r in results])),
            'avg_psnr_y': float(np.mean([r['psnr_y'] for r in results])),
            'avg_ssim_y': float(np.mean([r['ssim_y'] for r in results])),
            'avg_mae': float(np.mean([r['mae'] for r in results])),
            'avg_mse': float(np.mean([r['mse'] for r in results]))
        }
        avg_metrics.append(avg_metrics_total)
        latex_string += f"{avg_metrics_total['avg_psnr']:.2f} & {avg_metrics_total['avg_ssim']:.4f} & {avg_metrics_total['avg_psnr_y']:.2f} & {avg_metrics_total['avg_ssim_y']:.4f} & {avg_metrics_total['avg_mae']:.2f} & {avg_metrics_total['avg_mse']:.2f}"
        print(latex_string)
        with open(os.path.join(self.output_dir, 'avg_metrics.json'), 'w') as f:
            json.dump(avg_metrics, f, indent=2)
        return results


# Helper function to load DehazeFormer model
def load_dehazeformer_model(model_name, checkpoint_path):
    """
    Load DehazeFormer model
    :param model_name: 'dehazeformer-s', 'dehazeformer-b', 'dehazeformer-l', etc.
    :param checkpoint_path: path to the .pth checkpoint file
    """
    import sys
    sys.path.append('/content/DehazeFormer')
    from models import dehazeformer_s, dehazeformer_b, dehazeformer_l

    # Create model
    model_dict = {
        'dehazeformer-s': dehazeformer_s,
        'dehazeformer-b': dehazeformer_b,
        'dehazeformer-l': dehazeformer_l
    }

    network = model_dict[model_name]()
    network.cuda()

    # Load checkpoint
    state_dict = torch.load(checkpoint_path)['state_dict']
    new_state_dict = OrderedDict()

    # Remove 'module.' prefix if present
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        new_state_dict[name] = v

    network.load_state_dict(new_state_dict)
    network.eval()

    return network


model_name = 'dehazeformer-b'
# Load DehazeFormer model
folders = ['indoor', 'outdoor','reside6k','rshaze']
for folder in folders:
  print(folder)
  model = load_dehazeformer_model(
      model_name=model_name,  # or 'dehazeformer-b', 'dehazeformer-l'
      checkpoint_path=f'/content/DehazeFormer/saved_models/{folder}/{model_name}.pth'
  )

  # Create evaluation dataset
  eval_dataset = EvalDataset(
      corrupted_dir='/content/drive/MyDrive/Chris/data/eval_dataset/corrupted',
      original_dir='/content/drive/MyDrive/Chris/data/image_test',
      mask_dir='/content/drive/MyDrive/Chris/data/mask',
      metadata_path='/content/drive/MyDrive/Chris/data/eval_dataset/metadata.json'
  )


  # Run evaluation
  evaluator = DehazeFormerEvaluator(
      model=model,
      eval_dataset=eval_dataset,
      output_dir='/content/drive/MyDrive/Chris/results',
      task=f"{folder}_{model_name}"
  )

  results = evaluator.evaluate()

indoor


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|██████████| 300/300 [07:27<00:00,  1.49s/it]



=== Evaluation Results ===
Average PSNR (RGB): 20.26 dB
Average SSIM (RGB): 0.7778
Average PSNR (Y):   21.89 dB
Average SSIM (Y):   0.8045
Average MAE:        25.43
Average MSE:        1648.37
======= Alpha [0.1, 0.5] =======
PSNR (RGB): 25.97
SSIM (RGB): 0.9204
PSNR (Y):   27.62
SSIM (Y):   0.9367
MAE:        11.55
MSE:        284.72
======= Alpha [0.5, 0.8] =======
PSNR (RGB): 19.52
SSIM (RGB): 0.7707
PSNR (Y):   21.16
SSIM (Y):   0.8008
MAE:        24.45
MSE:        1308.45
======= Alpha [0.8, 1.0] =======
PSNR (RGB): 15.27
SSIM (RGB): 0.6423
PSNR (Y):   16.90
SSIM (Y):   0.6761
MAE:        40.28
MSE:        3351.95
25.97 & 0.9204 & 27.62 & 0.9367 & 11.55 & 284.72 &19.52 & 0.7707 & 21.16 & 0.8008 & 24.45 & 1308.45 &15.27 & 0.6423 & 16.90 & 0.6761 & 40.28 & 3351.95 &20.26 & 0.7778 & 21.89 & 0.8045 & 25.43 & 1648.37
outdoor


100%|██████████| 300/300 [03:09<00:00,  1.58it/s]



=== Evaluation Results ===
Average PSNR (RGB): 19.67 dB
Average SSIM (RGB): 0.7695
Average PSNR (Y):   21.41 dB
Average SSIM (Y):   0.8011
Average MAE:        30.82
Average MSE:        2228.06
======= Alpha [0.1, 0.5] =======
PSNR (RGB): 27.43
SSIM (RGB): 0.9202
PSNR (Y):   29.35
SSIM (Y):   0.9401
MAE:        11.42
MSE:        311.06
======= Alpha [0.5, 0.8] =======
PSNR (RGB): 17.81
SSIM (RGB): 0.7567
PSNR (Y):   19.48
SSIM (Y):   0.7929
MAE:        32.12
MSE:        2025.99
======= Alpha [0.8, 1.0] =======
PSNR (RGB): 13.75
SSIM (RGB): 0.6315
PSNR (Y):   15.40
SSIM (Y):   0.6704
MAE:        48.91
MSE:        4347.12
27.43 & 0.9202 & 29.35 & 0.9401 & 11.42 & 311.06 &17.81 & 0.7567 & 19.48 & 0.7929 & 32.12 & 2025.99 &13.75 & 0.6315 & 15.40 & 0.6704 & 48.91 & 4347.12 &19.67 & 0.7695 & 21.41 & 0.8011 & 30.82 & 2228.06
reside6k


100%|██████████| 300/300 [03:09<00:00,  1.58it/s]



=== Evaluation Results ===
Average PSNR (RGB): 19.85 dB
Average SSIM (RGB): 0.7698
Average PSNR (Y):   21.49 dB
Average SSIM (Y):   0.8001
Average MAE:        29.53
Average MSE:        2031.33
======= Alpha [0.1, 0.5] =======
PSNR (RGB): 27.49
SSIM (RGB): 0.9209
PSNR (Y):   29.18
SSIM (Y):   0.9386
MAE:        10.85
MSE:        260.11
======= Alpha [0.5, 0.8] =======
PSNR (RGB): 18.18
SSIM (RGB): 0.7588
PSNR (Y):   19.79
SSIM (Y):   0.7938
MAE:        30.19
MSE:        1739.35
======= Alpha [0.8, 1.0] =======
PSNR (RGB): 13.88
SSIM (RGB): 0.6296
PSNR (Y):   15.50
SSIM (Y):   0.6680
MAE:        47.56
MSE:        4094.52
27.49 & 0.9209 & 29.18 & 0.9386 & 10.85 & 260.11 &18.18 & 0.7588 & 19.79 & 0.7938 & 30.19 & 1739.35 &13.88 & 0.6296 & 15.50 & 0.6680 & 47.56 & 4094.52 &19.85 & 0.7698 & 21.49 & 0.8001 & 29.53 & 2031.33
rshaze


100%|██████████| 300/300 [03:10<00:00,  1.58it/s]


=== Evaluation Results ===
Average PSNR (RGB): 16.33 dB
Average SSIM (RGB): 0.7268
Average PSNR (Y):   18.02 dB
Average SSIM (Y):   0.7694
Average MAE:        36.80
Average MSE:        2394.24
======= Alpha [0.1, 0.5] =======
PSNR (RGB): 20.61
SSIM (RGB): 0.8676
PSNR (Y):   22.50
SSIM (Y):   0.8984
MAE:        20.83
MSE:        805.48
======= Alpha [0.5, 0.8] =======
PSNR (RGB): 15.23
SSIM (RGB): 0.7084
PSNR (Y):   16.85
SSIM (Y):   0.7545
MAE:        39.55
MSE:        2481.15
======= Alpha [0.8, 1.0] =======
PSNR (RGB): 13.14
SSIM (RGB): 0.6045
PSNR (Y):   14.73
SSIM (Y):   0.6552
MAE:        50.01
MSE:        3896.08
20.61 & 0.8676 & 22.50 & 0.8984 & 20.83 & 805.48 &15.23 & 0.7084 & 16.85 & 0.7545 & 39.55 & 2481.15 &13.14 & 0.6045 & 14.73 & 0.6552 & 50.01 & 3896.08 &16.33 & 0.7268 & 18.02 & 0.7694 & 36.80 & 2394.24
